In [0]:
%sql
create database auroradb

## Ingest to Bronze table

In [0]:
# Notebook: 01_Bronze_Ingestion
from pyspark.sql.functions import current_timestamp, col

def ingest_to_bronze(file_path, table_name):
    df = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .load(file_path)
    
    # Thêm metadata: thời gian nạp và tên file nguồn
    df = df.withColumn("ingestion_timestamp", current_timestamp()) \
           .withColumn("source_file", col('_metadata.file_path'))
    
    df.write.format("delta").mode("overwrite").saveAsTable(f"auroradb.{table_name}")

# Thực hiện nạp
ingest_to_bronze("/Volumes/workspace/auroradb/raw/users_data.csv", "users_raw")
ingest_to_bronze("/Volumes/workspace/auroradb/raw/cards_data.csv", "cards_raw")
ingest_to_bronze("/Volumes/workspace/auroradb/raw/transactions_data.csv", "transactions_raw")
ingest_to_bronze("/Volumes/workspace/auroradb/raw/mcc_codes.csv", "mcc_raw")

## SILVER: Cleaning & Star Schema 

In [0]:
%sql
select * from auroradb.users_raw limit 10

In [0]:
# Notebook: 02_Silver_Standardization
from pyspark.sql.functions import col, regexp_replace, to_date, date_format, year, month, dayofmonth, quarter, dayofweek, when, min, max, explode, sequence

def clean_money(column):
    return regexp_replace(col(column), "[\\$,]", "").cast("double")

# --- 2.1 Dim_Users ---
df_users = spark.table("auroradb.users_raw") \
    .withColumn("yearly_income", clean_money("yearly_income")) \
    .withColumn("total_debt", clean_money("total_debt")) \
    .withColumnRenamed("id", "user_id")
df_users.write.format("delta").mode("overwrite").saveAsTable("auroradb.dim_users")

# --- 2.2 Dim_Cards ---
df_cards = spark.table("auroradb.cards_raw") \
    .withColumn("credit_limit", clean_money("credit_limit")) \
    .withColumnRenamed("id", "card_id") \
    .withColumnRenamed("client_id", "user_id")
df_cards.write.format("delta").mode("overwrite").saveAsTable("auroradb.dim_cards")

# --- 2.3 Dim_Merchants & Fact_Transactions ---
df_trans_raw = spark.table("auroradb.transactions_raw")
df_trans_cleaned = df_trans_raw \
    .withColumn("amount", clean_money("amount")) \
    .withColumn("date_dt", to_date(col("date"), "M/d/yyyy H:mm")) \
    .withColumn("date_key", date_format(col("date_dt"), "yyyyMMdd").cast("int"))

# Tách Dim_Merchants
df_merchants = df_trans_cleaned.select("merchant_id", "merchant_city", "merchant_state", "zip").distinct()
df_merchants.write.format("delta").mode("overwrite").saveAsTable("auroradb.dim_merchants")

# Fact_Transactions
fact_transactions = df_trans_cleaned.select(
    col("id").alias("transaction_id"), "date_key", col("client_id").alias("user_id"), 
    "card_id", "merchant_id", col("mcc").alias("mcc_id"), "amount", "use_chip", "errors"
)
fact_transactions.write.format("delta").mode("overwrite").saveAsTable("auroradb.fact_transactions")

# --- 2.4 Dim_Date ---
# (Tương tự logic tạo sequence ngày đã viết ở trên, lưu vào silver.dim_date)

def clean_currency(df, column_name):
    """Hàm làm sạch ký tự $ và chuyển sang kiểu Double"""
    return df.withColumn(column_name, regexp_replace(col(column_name), "[\\$,]", "").cast("double"))

trans_cleaned = clean_currency(df_trans_raw, "amount")
trans_cleaned = trans_cleaned.withColumn("temp_timestamp", to_date(col("date"), "M/d/yyyy H:mm"))
trans_cleaned = trans_cleaned.withColumn("date_key", date_format(col("temp_timestamp"), "yyyyMMdd").cast("int"))

date_bounds = trans_cleaned.select(min("temp_timestamp"), max("temp_timestamp")).collect()[0]
start_date, end_date = date_bounds[0], date_bounds[1]

dim_date = spark.sql(f"SELECT sequence(to_date('{start_date}'), to_date('{end_date}'), interval 1 day) as date_array") \
    .withColumn("full_date", explode("date_array")) \
    .select(
        date_format(col("full_date"), "yyyyMMdd").cast("int").alias("date_key"),
        "full_date",
        dayofmonth(col("full_date")).alias("day"),
        month(col("full_date")).alias("month"),
        year(col("full_date")).alias("year"),
        quarter(col("full_date")).alias("quarter"),
        date_format(col("full_date"), "EEEE").alias("day_of_week"),
        when(dayofweek(col("full_date")).isin(1, 7), True).otherwise(False).alias("is_weekend")
    )

dim_date.write.format("delta").mode("overwrite").saveAsTable("auroradb.dim_date")


In [0]:
# Xử lý distinct merchant_id
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

# Định nghĩa "cửa sổ" dựa trên merchant_id
window_spec = Window.partitionBy("merchant_id").orderBy("merchant_city")

# Lấy duy nhất 1 dòng cho mỗi ID
df_merchants = spark.table("auroradb.dim_merchants")
df_merchants_unique = df_merchants.withColumn("row_num", row_number().over(window_spec)) \
    .filter("row_num = 1") \
    .drop("row_num")

df_merchants_unique.write.format("delta").mode("overwrite").saveAsTable("auroradb.dim_merchants_unique")

## GOLD: Business Insights

In [0]:
# Notebook: 03_Gold_Business_Logic

# --- 3.1 Sức khỏe Tài chính & Chỉ số DTI ---
# Giả định nợ hàng tháng = 5% tổng nợ (theo tiêu chuẩn ngân hàng)
df_gold_risk = spark.table("auroradb.dim_users") \
    .withColumn("monthly_income", col("yearly_income") / 12) \
    .withColumn("est_monthly_debt_payment", col("total_debt") * 0.05 / 12) \
    .withColumn("dti_ratio", (col("est_monthly_debt_payment") / col("monthly_income")) * 100) \
    .withColumn("risk_segment", 
        when(col("dti_ratio") > 43, "At-Risk Borrowers")
        .when(col("credit_score") > 740, "Sophisticated Opportunists")
        .otherwise("Standard"))

df_gold_risk.write.format("delta").mode("overwrite").saveAsTable("auroradb.customer_risk_analysis")

# --- 3.2 Phân tích Thế hệ Z (Gen Z) ---
# Gen Z: Sinh từ 1997 - 2012
df_genz = spark.table("auroradb.dim_users").filter("birth_year >= 1997 AND birth_year <= 2012") \
    .join(spark.table("auroradb.fact_transactions"), "user_id") \
    .join(spark.table("auroradb.mcc_raw"), "mcc_id") \
    .groupBy("description") \
    .sum("amount") \
    .withColumnRenamed("sum(amount)", "total_spent") \
    .orderBy(col("total_spent").desc())

df_genz.write.format("delta").mode("overwrite").saveAsTable("auroradb.genz_spending_trends")

# --- 3.3 Tối ưu hóa Merchant & Lỗi Giao dịch ---
df_merchant_errors = spark.table("auroradb.fact_transactions") \
    .filter("errors IS NOT NULL") \
    .groupBy("merchant_id", "errors") \
    .count() \
    .join(spark.table("auroradb.dim_merchants"), "merchant_id")

df_merchant_errors.write.format("delta").mode("overwrite").saveAsTable("auroradb.merchant_error_monitoring")